

<h1>📊 Feature Engineering for Kitchen Appliance IoT Logs</h1>
<p>Complete theory, math, examples, and Python code for creating new columns from IoT sensor logs.</p>

<!-- SECTION 1 -->
<div class="section">
<h2>1️⃣ Timestamp Features</h2>
<p><strong>Theory:</strong> Extracting features from timestamp helps detect usage patterns across hours, days, weeks, or months. Many appliances have peak usage times (daily or weekly).</p>
<p><strong>Math:</strong> None directly; these are categorical or cyclic features derived from datetime.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Example</th><th>Python Code</th></tr>
<tr>
<td>hour</td>
<td>Detect daily usage pattern</td>
<td>10 → 10 AM</td>
<td><code>df['hour'] = df.index.hour</code></td>
</tr>
<tr>
<td>day_of_week</td>
<td>Detect weekday/weekend patterns</td>
<td>0 → Monday</td>
<td><code>df['day_of_week'] = df.index.dayofweek</code></td>
</tr>
<tr>
<td>is_weekend</td>
<td>Binary flag for weekend usage</td>
<td>1 → Saturday/Sunday</td>
<td><code>df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)</code></td>
</tr>
<tr>
<td>month</td>
<td>Detect seasonal appliance usage</td>
<td>1 → January</td>
<td><code>df['month'] = df.index.month</code></td>
</tr>
<tr>
<td>day</td>
<td>Detect day-of-month patterns</td>
<td>15 → 15th day</td>
<td><code>df['day'] = df.index.day</code></td>
</tr>
<tr>
<td>week</td>
<td>Weekly aggregation patterns</td>
<td>3 → 3rd week</td>
<td><code>df['week'] = df.index.isocalendar().week</code></td>
</tr>
</table>

<div class="example">Example: Oven usage peaks at 7–9 PM (hour), Dishwasher usage peaks on weekends (is_weekend).</div>
</div>

<!-- SECTION 2 -->
<div class="section">
<h2>2️⃣ Device Usage Features</h2>
<p><strong>Theory:</strong> Capture real usage behavior, duration, energy consumption, and appliance status. Useful for maintenance and customer usage analysis.</p>
<p><strong>Math:</strong></p>
<ul>
<li><strong>Usage duration:</strong> difference between consecutive timestamps <code>duration = t_i - t_{i-1}</code></li>
<li><strong>Power consumption:</strong> energy = power × duration <code>kWh = P × t</code></li>
<li>Status and error flags are encoded numerically for modeling.</li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Example</th><th>Python Code</th></tr>
<tr>
<td>usage_duration</td>
<td>Time appliance is ON</td>
<td>60 seconds</td>
<td><code>df['usage_duration'] = df.index.to_series().diff().dt.seconds.fillna(0)</code></td>
</tr>
<tr>
<td>power_consumption</td>
<td>Energy consumed per interval</td>
<td>180 W × 60 sec = 3 Wh</td>
<td><code>df['power_consumption'] = df['power'] * df['usage_duration']</code></td>
</tr>
<tr>
<td>status_flag</td>
<td>Encode categorical status numerically</td>
<td>RUNNING → 2</td>
<td><code>df['status_flag'] = df['status'].map({'OFF':0,'IDLE':1,'RUNNING':2})</code></td>
</tr>
<tr>
<td>error_flag</td>
<td>Binary flag for errors</td>
<td>1 if error_code not null</td>
<td><code>df['error_flag'] = df['error_code'].notna().astype(int)</code></td>
</tr>
</table>

<div class="example">Example: Coffee machine short bursts → usage_duration identifies real brewing events. Oven logs “RUNNING” → status_flag=2.</div>
</div>

<!-- SECTION 3 -->
<div class="section">
<h2>3️⃣ Rolling / Aggregation Features</h2>
<p><strong>Theory:</strong> Time-series logs often have noise. Rolling averages smooth readings. Differences detect trends or sudden spikes.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Moving average (MA): <code>MA_t = (x_t + x_{t-1} + ... + x_{t-k+1}) / k</code></li>
<li>Difference: <code>diff_t = x_t - x_{t-1}</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_MA_5</td>
<td>5-reading moving average of temperature</td>
<td><code>df['temp_MA_5'] = df['temperature'].rolling(5).mean()</code></td>
</tr>
<tr>
<td>power_MA_10</td>
<td>10-reading moving average of power</td>
<td><code>df['power_MA_10'] = df['power'].rolling(10).mean()</code></td>
</tr>
<tr>
<td>temp_diff</td>
<td>Change from previous temperature reading</td>
<td><code>df['temp_diff'] = df['temperature'].diff()</code></td>
</tr>
<tr>
<td>power_diff</td>
<td>Change in power reading</td>
<td><code>df['power_diff'] = df['power'].diff()</code></td>
</tr>
</table>

<div class="example">Example: Oven temperature fluctuates ±5°C → temp_MA_5 smooths trend. Power spike → power_diff highlights anomaly.</div>
</div>

<!-- SECTION 4 -->
<div class="section">
<h2>4️⃣ Anomaly / Outlier Features</h2>
<p><strong>Theory:</strong> Detect abnormal appliance behavior using statistics. Z-score and IQR methods are commonly used.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Z-score: <code>z_i = (x_i - μ)/σ</code>, flag if |z_i|>3</li>
<li>IQR: <code>Q1 - 1.5*IQR &lt; x_i &gt; Q3 + 1.5*IQR</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_zscore</td>
<td>Detect temperature spikes</td>
<td><code>df['temp_zscore'] = (df['temperature'] - df['temperature'].mean())/df['temperature'].std()</code></td>
</tr>
<tr>
<td>power_zscore</td>
<td>Detect unusual power consumption</td>
<td><code>df['power_zscore'] = (df['power'] - df['power'].mean())/df['power'].std()</code></td>
</tr>
<tr>
<td>is_temp_outlier</td>
<td>Binary flag for temp outliers</td>
<td><code>df['is_temp_outlier'] = (df['temp_zscore'].abs() > 3).astype(int)</code></td>
</tr>
</table>

<div class="example">Example: Oven temperature jumps from 180°C → 250°C → flagged as outlier.</div>
</div>

<!-- SECTION 5 -->
<div class="section">
<h2>5️⃣ Lag / Cumulative Features</h2>
<p><strong>Theory:</strong> Capture trends over time and cumulative usage.</p>
<p><strong>Math:</strong></p>
<ul>
<li>Lag: <code>lag_t = x_t - x_{t-1}</code></li>
<li>Cumulative sum: <code>cumsum_t = Σ x_i</code></li>
<li>Rolling max: <code>max_t = max(x_t, x_{t-1}, ..., x_{t-k+1})</code></li>
</ul>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>temp_lag1</td>
<td>Previous reading of temperature</td>
<td><code>df['temp_lag1'] = df['temperature'].shift(1)</code></td>
</tr>
<tr>
<td>power_cumsum</td>
<td>Cumulative power usage</td>
<td><code>df['power_cumsum'] = df['power'].cumsum()</code></td>
</tr>
<tr>
<td>rolling_max_power</td>
<td>Max power over last 10 readings</td>
<td><code>df['rolling_max_power'] = df['power'].rolling(10).max()</code></td>
</tr>
</table>
</div>

<!-- SECTION 6 -->
<div class="section">
<h2>6️⃣ Duplicate / Missing Value Features</h2>
<p><strong>Theory:</strong> Flag quality issues in logs for data cleaning or modeling.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>is_duplicate</td>
<td>Row is a duplicate</td>
<td><code>df['is_duplicate'] = df.duplicated().astype(int)</code></td>
</tr>
<tr>
<td>missing_flag_temp</td>
<td>Temperature missing value</td>
<td><code>df['missing_flag_temp'] = df['temperature'].isna().astype(int)</code></td>
</tr>
</table>

<div class="note">Duplicate spikes or missing values often indicate hardware or network issues.</div>
</div>

<!-- SECTION 7 -->
<div class="section">
<h2>✅ Key Takeaways</h2>
<ul>
<li>Timestamp features capture usage trends (hour, day, weekend, week).</li>
<li>Device usage features quantify appliance behavior (duration, energy, status).</li>
<li>Rolling and difference features smooth logs and detect sudden changes.</li>
<li>Anomaly/outlier features identify abnormal behavior using Z-score or IQR.</li>
<li>Lag and cumulative features capture trends over time.</li>
<li>Data quality flags (`is_duplicate`, `missing_flag`) ensure reliable analysis.</li>
<li>Feature engineering transforms raw IoT logs into actionable insights for analytics, maintenance, and customer usage behavior.</li>
</ul>
</div>

</body>
</html>


=================================================================================================================================================

<h1>📊 Generic Feature Engineering Cheat Sheet</h1>
<p>Reference for creating new features from any tabular dataset.</p>

<!-- SECTION 1 -->
<div class="section">
<h2>1️⃣ Timestamp Features</h2>
<p><strong>Theory:</strong> Extract time components to capture patterns in daily, weekly, monthly, or seasonal cycles.</p>
<p><strong>Math:</strong> Derived categorical/cyclic features from datetime.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Example</th><th>Python Code</th></tr>
<tr>
<td>hour</td><td>Daily pattern</td><td>14 → 2 PM</td><td><code>df['hour'] = df['timestamp'].dt.hour</code></td>
</tr>
<tr>
<td>day_of_week</td><td>Weekday/weekend patterns</td><td>0 → Monday</td><td><code>df['day_of_week'] = df['timestamp'].dt.dayofweek</code></td>
</tr>
<tr>
<td>is_weekend</td><td>Binary weekend flag</td><td>1 → Sat/Sun</td><td><code>df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)</code></td>
</tr>
<tr>
<td>month</td><td>Seasonal pattern</td><td>3 → March</td><td><code>df['month'] = df['timestamp'].dt.month</code></td>
</tr>
<tr>
<td>day</td><td>Day of month</td><td>21 → 21st day</td><td><code>df['day'] = df['timestamp'].dt.day</code></td>
</tr>
<tr>
<td>week</td><td>Week number</td><td>10 → 10th week</td><td><code>df['week'] = df['timestamp'].dt.isocalendar().week</code></td>
</tr>
</table>
<div class="example">Example: Sales data – detect weekday vs weekend purchase patterns.</div>
</div>

<!-- SECTION 2 -->
<div class="section">
<h2>2️⃣ Categorical Features</h2>
<p><strong>Theory:</strong> Encode categorical columns for ML models and analysis.</p>
<p><strong>Math:</strong> One-hot encoding, label encoding, frequency encoding.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>label_encoded</td><td>Convert categories to numbers</td>
<td><code>from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['category_encoded'] = le.fit_transform(df['category'])</code></td>
</tr>
<tr>
<td>one_hot</td><td>Create binary columns</td>
<td><code>df = pd.get_dummies(df, columns=['category'])</code></td>
</tr>
<tr>
<td>frequency_encoded</td><td>Encode by frequency</td>
<td><code>freq = df['category'].value_counts()
df['category_freq'] = df['category'].map(freq)</code></td>
</tr>
</table>
<div class="example">Example: Product categories encoded for prediction of sales or churn.</div>
</div>

<!-- SECTION 3 -->
<div class="section">
<h2>3️⃣ Numerical Features</h2>
<p><strong>Theory:</strong> Transform numerical features to normalize, detect trends, or scale for models.</p>
<p><strong>Math:</strong> Standardization (Z-score), normalization (Min-Max), log transformation.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>z_score</td><td>Standardize numeric column</td>
<td><code>df['num_z'] = (df['num'] - df['num'].mean())/df['num'].std()</code></td>
</tr>
<tr>
<td>min_max</td><td>Scale between 0 and 1</td>
<td><code>df['num_norm'] = (df['num'] - df['num'].min())/(df['num'].max()-df['num'].min())</code></td>
</tr>
<tr>
<td>log_transform</td><td>Reduce skewness</td>
<td><code>import numpy as np
df['num_log'] = np.log1p(df['num'])</code></td>
</tr>
<tr>
<td>diff</td><td>Difference with previous row</td>
<td><code>df['num_diff'] = df['num'].diff()</code></td>
</tr>
<tr>
<td>cumsum</td><td>Cumulative sum</td>
<td><code>df['num_cumsum'] = df['num'].cumsum()</code></td>
</tr>
<tr>
<td>rolling_mean</td><td>Moving average</td>
<td><code>df['num_ma5'] = df['num'].rolling(5).mean()</code></td>
</tr>
</table>
<div class="example">Example: Customer purchase amounts – normalize, log transform, and compute rolling averages.</div>
</div>

<!-- SECTION 4 -->
<div class="section">
<h2>4️⃣ Interaction / Derived Features</h2>
<p><strong>Theory:</strong> Combine features to create new signals that may improve predictions.</p>
<p><strong>Math:</strong> Arithmetic combinations, ratios, or boolean logic.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>ratio_feature</td><td>Feature ratio</td>
<td><code>df['ratio'] = df['feature1']/df['feature2']</code></td>
</tr>
<tr>
<td>sum_feature</td><td>Feature sum</td>
<td><code>df['sum_feat'] = df['feature1'] + df['feature2']</code></td>
</tr>
<tr>
<td>binary_flag</td><td>Binary flag based on condition</td>
<td><code>df['flag'] = (df['feature'] &gt; 0).astype(int)</code></td>
</tr>
</table>
<div class="example">Example: Total spending = price × quantity, High-value customer flag = 1 if spending > 100.</div>
</div>

<!-- SECTION 5 -->
<div class="section">
<h2>5️⃣ Missing & Duplicate Features</h2>
<p><strong>Theory:</strong> Identify and flag data quality issues before modeling.</p>

<table>
<tr><th>Column</th><th>Purpose</th><th>Python Code</th></tr>
<tr>
<td>is_duplicate</td><td>Flag duplicates</td>
<td><code>df['is_duplicate'] = df.duplicated().astype(int)</code></td>
</tr>
<tr>
<td>missing_flag</td><td>Flag missing values</td>
<td><code>df['missing_flag'] = df['feature'].isna().astype(int)</code></td>
</tr>
</table>
<div class="example">Example: Flag rows missing customer email or duplicate transaction entries.</div>
</div>

<!-- SECTION 6 -->
<div class="section">
<h2>6️⃣ Key Takeaways</h2>
<ul>
<li>Timestamp features capture temporal patterns.</li>
<li>Categorical encoding is required for ML models.</li>
<li>Numerical transformations normalize distributions and highlight trends.</li>
<li>Interaction features create new meaningful signals.</li>
<li>Missing & duplicate flags ensure data quality.</li>
<li>Rolling, diff, cumulative, and log transforms are useful for trend analysis and anomaly detection.</li>
</ul>
</div>

</body>
</html>
